In [15]:
# Cell 2 — Ingredient Database
INGREDIENT_DB = {
    "Toned Milk 3%": {
        "fat_pct": 3.0, "msnf_pct": 8.5, "sugars_pct": 4.8,
        "water_pct": 87.7, "category": "dairy", "locked": False,
        "note": "Standard toned milk"
    },
    "Cream 25%": {
        "fat_pct": 25.0, "msnf_pct": 6.5, "sugars_pct": 3.0,
        "water_pct": 64.0, "category": "dairy", "locked": False,
        "note": "Dairy cream 25% fat"
    },
    "Skimmed Milk Powder": {
        "fat_pct": 0.1, "msnf_pct": 95.0, "sugars_pct": 51.0,
        "water_pct": 3.5, "category": "dairy_powder", "locked": False,
        "note": "SMP — high MSNF source"
    },
    "Condensed Milk Nestle": {
        "fat_pct": 8.0, "msnf_pct": 20.0, "sugars_pct": 55.0,
        "water_pct": 27.0, "category": "dairy", "locked": False,
        "note": "Sweetened condensed milk"
    },
    "Sucrose/sugar": {
        "fat_pct": 0.0, "msnf_pct": 0.0, "sugars_pct": 100.0,
        "water_pct": 0.0, "category": "sugar", "locked": False,
        "note": "Table sugar"
    },
    "Dextrose monohydrate": {
        "fat_pct": 0.0, "msnf_pct": 0.0, "sugars_pct": 91.0,
        "water_pct": 9.0, "category": "sugar", "locked": False,
        "note": "Dextrose mono"
    },
    "Glucose Syrup (40-42DE)": {
        "fat_pct": 0.0, "msnf_pct": 0.0, "sugars_pct": 78.0,
        "water_pct": 22.0, "category": "sugar", "locked": False,
        "note": "Glucose syrup 40-42 DE"
    },
    "Stabilizer": {
        "fat_pct": 0.0, "msnf_pct": 0.0, "sugars_pct": 0.0,
        "water_pct": 5.0, "category": "stabilizer", "locked": True,
        "note": "Stabilizer blend — LOCKED"
    }
}

print(f"Loaded {len(INGREDIENT_DB)} ingredients into INGREDIENT_DB")
for name, props in INGREDIENT_DB.items():
    lock_tag = ' 🔒' if props['locked'] else ''
    print(f"  • {name}: fat={props['fat_pct']}%, msnf={props['msnf_pct']}%, sugars={props['sugars_pct']}%{lock_tag}")

Loaded 8 ingredients into INGREDIENT_DB
  • Toned Milk 3%: fat=3.0%, msnf=8.5%, sugars=4.8%
  • Cream 25%: fat=25.0%, msnf=6.5%, sugars=3.0%
  • Skimmed Milk Powder: fat=0.1%, msnf=95.0%, sugars=51.0%
  • Condensed Milk Nestle: fat=8.0%, msnf=20.0%, sugars=55.0%
  • Sucrose/sugar: fat=0.0%, msnf=0.0%, sugars=100.0%
  • Dextrose monohydrate: fat=0.0%, msnf=0.0%, sugars=91.0%
  • Glucose Syrup (40-42DE): fat=0.0%, msnf=0.0%, sugars=78.0%
  • Stabilizer: fat=0.0%, msnf=0.0%, sugars=0.0% 🔒


In [16]:
# Cell 3 — Pydantic Models
from pydantic import BaseModel, Field
from typing import Optional

class IngredientEntry(BaseModel):
    fat_pct: float = 0.0
    msnf_pct: float = 0.0
    sugars_pct: float = 0.0
    water_pct: float = 0.0
    category: str = 'other'
    locked: bool = False
    note: str = ''

class ProductionTargets(BaseModel):
    lossPct: float = Field(ge=0, le=20)
    mixDensity: float = Field(ge=0.9, le=1.2)
    overrunPct: float = Field(ge=0, le=150)
    skuSizeLiters: float = Field(gt=0)
    targetVolumeLiters: float = Field(gt=0)

class OptimizationTargets(BaseModel):
    fat_pct: Optional[float] = None
    msnf_pct: Optional[float] = None
    sugars_pct: Optional[float] = None

class BatchMetrics(BaseModel):
    gross_liquid_volume_L: float
    batch_mass_g: float
    n_skus: int
    scale_factor: float

class RecipeMetrics(BaseModel):
    total_mass_g: float
    fat_pct: float
    msnf_pct: float
    sugars_pct: float
    water_pct: float
    total_solids_pct: float

print('✅ Pydantic models loaded.')

✅ Pydantic models loaded.


In [17]:
# Cell 4 — Batch Sizing (Production Math)
import math

def compute_batch_sizing(recipe: list, targets: ProductionTargets) -> BatchMetrics:
    """
    Mirrors Level 1-3 production planning engine exactly.
    pre_overrun_volume_L  = targetVolumeLiters / (1 + overrunPct / 100)
    gross_liquid_volume_L = pre_overrun_volume_L / (1 - lossPct / 100)
    batch_mass_g          = gross_liquid_volume_L * 1000 * mixDensity
    n_skus                = ceil(targetVolumeLiters / skuSizeLiters)
    scale_factor          = batch_mass_g / sum(recipe quantities)
    """
    pre_overrun_volume_L = targets.targetVolumeLiters / (1 + targets.overrunPct / 100)
    gross_liquid_volume_L = pre_overrun_volume_L / (1 - targets.lossPct / 100)
    batch_mass_g = gross_liquid_volume_L * 1000 * targets.mixDensity
    n_skus = math.ceil(targets.targetVolumeLiters / targets.skuSizeLiters)
    recipe_total_g = sum(item['quantity_g'] for item in recipe)
    scale_factor = batch_mass_g / recipe_total_g

    return BatchMetrics(
        gross_liquid_volume_L=round(gross_liquid_volume_L, 2),
        batch_mass_g=round(batch_mass_g, 2),
        n_skus=n_skus,
        scale_factor=round(scale_factor, 6)
    )

print('✅ compute_batch_sizing() ready.')

✅ compute_batch_sizing() ready.


In [18]:
# Cell 5 — Recipe Metrics Calculator
def compute_recipe_metrics(recipe: dict) -> RecipeMetrics:
    """Given {ingredient_name: grams}, compute nutritional metrics."""
    total = sum(recipe.values())
    if total == 0:
        return RecipeMetrics(total_mass_g=0, fat_pct=0, msnf_pct=0,
                             sugars_pct=0, water_pct=0, total_solids_pct=0)
    fat_g = msnf_g = sugars_g = water_g = 0.0
    for name, grams in recipe.items():
        ing = INGREDIENT_DB.get(name)
        if not ing:
            continue
        fat_g    += grams * ing['fat_pct'] / 100
        msnf_g   += grams * ing['msnf_pct'] / 100
        sugars_g += grams * ing['sugars_pct'] / 100
        water_g  += grams * ing['water_pct'] / 100

    return RecipeMetrics(
        total_mass_g=round(total, 2),
        fat_pct=round(fat_g / total * 100, 3),
        msnf_pct=round(msnf_g / total * 100, 3),
        sugars_pct=round(sugars_g / total * 100, 3),
        water_pct=round(water_g / total * 100, 3),
        total_solids_pct=round((1 - water_g / total) * 100, 3)
    )

print('✅ compute_recipe_metrics() ready.')

✅ compute_recipe_metrics() ready.


In [19]:
# Cell 6 — LP Optimizer (PuLP / CBC)
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, PULP_CBC_CMD, LpStatus, value

SUGAR_BOUNDS = {
    'gelato':    {'sucrose_max_pct': 22, 'dextrose_max_pct': 8, 'glucose_max_pct': 8},
    'ice_cream': {'sucrose_max_pct': 22, 'dextrose_max_pct': 8, 'glucose_max_pct': 8},
    'kulfi':     {'sucrose_max_pct': 20, 'dextrose_max_pct': 6, 'glucose_max_pct': 6},
    'sorbet':    {'sucrose_max_pct': 25, 'dextrose_max_pct': 15, 'glucose_max_pct': 12},
}

DEVIATION_WEIGHT = 10.0
MOVEMENT_WEIGHT  = 0.001

def run_lp_optimizer(scaled_recipe, opt_targets, mode='gelato', mass_tolerance=0.10):
    """LP solver ported from balanceRecipeLP() in optimize_balancer_v2.ts."""
    names = list(scaled_recipe.keys())
    initial = list(scaled_recipe.values())
    total_W = sum(initial)
    n = len(names)
    warnings = []
    prob = LpProblem('ReverseEngine_LP', LpMinimize)
    x, pos, neg = [], [], []
    bounds = SUGAR_BOUNDS.get(mode, SUGAR_BOUNDS['gelato'])

    for i, name in enumerate(names):
        ing = INGREDIENT_DB.get(name, {})
        locked = ing.get('locked', False)
        if locked:
            lo, hi = initial[i], initial[i]
        else:
            lo = 0
            hi = max(initial[i] * 5, 1500)
            nm_lower = name.lower()
            if 'sucrose' in nm_lower or nm_lower == 'sucrose/sugar':
                hi = min(hi, total_W * bounds['sucrose_max_pct'] / 100)
            elif 'dextrose' in nm_lower:
                hi = min(hi, total_W * bounds['dextrose_max_pct'] / 100)
            elif 'glucose' in nm_lower:
                hi = min(hi, total_W * bounds['glucose_max_pct'] / 100)

        xi = LpVariable(f'x_{i}', lowBound=lo, upBound=hi)
        pi = LpVariable(f'pos_{i}', lowBound=0)
        ni = LpVariable(f'neg_{i}', lowBound=0)
        x.append(xi); pos.append(pi); neg.append(ni)

        if not locked:
            prob += xi - initial[i] == pi - ni, f'move_{i}'
        else:
            prob += pi == 0, f'pos_lock_{i}'
            prob += ni == 0, f'neg_lock_{i}'

    fat_over   = LpVariable('fat_over', lowBound=0)
    fat_under  = LpVariable('fat_under', lowBound=0)
    msnf_over  = LpVariable('msnf_over', lowBound=0)
    msnf_under = LpVariable('msnf_under', lowBound=0)
    sug_over   = LpVariable('sug_over', lowBound=0)
    sug_under  = LpVariable('sug_under', lowBound=0)

    free_idx = [i for i in range(n) if not INGREDIENT_DB.get(names[i], {}).get('locked', False)]
    obj = MOVEMENT_WEIGHT * lpSum(pos[i] + neg[i] for i in free_idx)
    if opt_targets.fat_pct is not None:
        obj += DEVIATION_WEIGHT * (fat_over + fat_under)
    if opt_targets.msnf_pct is not None:
        obj += DEVIATION_WEIGHT * (msnf_over + msnf_under)
    if opt_targets.sugars_pct is not None:
        obj += DEVIATION_WEIGHT * (sug_over + sug_under)
    prob += obj, 'total_cost'

    prob += lpSum(x) >= total_W * (1 - mass_tolerance), 'mass_lower'
    prob += lpSum(x) <= total_W * (1 + mass_tolerance), 'mass_upper'

    def coeff(i, field):
        return INGREDIENT_DB.get(names[i], {}).get(field, 0) / 100

    if opt_targets.fat_pct is not None:
        tg = (opt_targets.fat_pct / 100) * total_W
        prob += lpSum(x[i] * coeff(i, 'fat_pct') for i in range(n)) - fat_over + fat_under == tg, 'fat_eq'
    if opt_targets.msnf_pct is not None:
        tg = (opt_targets.msnf_pct / 100) * total_W
        prob += lpSum(x[i] * coeff(i, 'msnf_pct') for i in range(n)) - msnf_over + msnf_under == tg, 'msnf_eq'
    if opt_targets.sugars_pct is not None:
        tg = (opt_targets.sugars_pct / 100) * total_W
        prob += lpSum(x[i] * coeff(i, 'sugars_pct') for i in range(n)) - sug_over + sug_under == tg, 'sugars_eq'

    prob.solve(PULP_CBC_CMD(msg=0))
    status = LpStatus[prob.status]
    if status != 'Optimal':
        return {'success': False, 'solver_status': status,
                'proposed_recipe': dict(scaled_recipe),
                'warnings': [f'Solver status: {status}. Recipe unchanged.']}

    proposed = {names[i]: max(0, value(x[i])) for i in range(n)}
    proposed_sum = sum(proposed.values())
    if abs(proposed_sum - total_W) > 1.0:
        factor = total_W / proposed_sum
        proposed = {k: v * factor for k, v in proposed.items()}
        warnings.append(f'Rescaled by {factor:.6f} to correct drift')

    return {'success': True, 'solver_status': 'Optimal — Linear Programming (Simplex)',
            'proposed_recipe': proposed, 'warnings': warnings}

print('✅ run_lp_optimizer() ready.')

✅ run_lp_optimizer() ready.


In [20]:
# Cell 7 — Custom Instruction Parser
import re

def parse_instruction(instruction: str) -> OptimizationTargets:
    """Extract fat_pct, msnf_pct, sugars_pct from a plain-text instruction string."""
    text = instruction.lower().strip()
    targets = {}
    fat_match = re.search(r'fat\s*(?:to\s*)?([\d.]+)\s*%?', text)
    if fat_match:
        targets['fat_pct'] = float(fat_match.group(1))
    msnf_match = re.search(r'msnf\s*(?:to\s*)?([\d.]+)\s*%?', text)
    if msnf_match:
        targets['msnf_pct'] = float(msnf_match.group(1))
    sugars_match = re.search(r'sugars?\s*(?:to\s*)?([\d.]+)\s*%?', text)
    if sugars_match:
        targets['sugars_pct'] = float(sugars_match.group(1))
    return OptimizationTargets(**targets)

_test = parse_instruction('Increase fat to 8%, MSNF to 10%, sugars to 20%')
print(f'✅ parse_instruction() ready.  Test parse: {_test}')

✅ parse_instruction() ready.  Test parse: fat_pct=8.0 msnf_pct=10.0 sugars_pct=20.0


In [21]:
# Cell 8 — Optimizer Tool (deterministic LP math, no AI)
def optimizer_tool(recipe: list, production_targets: dict,
                   custom_instruction: str, mode: str = 'gelato') -> dict:
    """
    Deterministic optimizer tool.
    Steps: batch sizing → scale recipe → parse targets → LP solve → diff.
    Returns a dict with all optimization results.
    """
    warnings = []

    # 1. Batch sizing
    prod = ProductionTargets(**production_targets)
    batch = compute_batch_sizing(recipe, prod)

    # 2. Scale recipe to batch
    scaled = {}
    for item in recipe:
        name = item['ingredient']
        if name not in INGREDIENT_DB:
            warnings.append(f"⚠️ '{name}' not in INGREDIENT_DB")
        scaled[name] = item['quantity_g'] * batch.scale_factor

    metrics_before = compute_recipe_metrics(scaled)

    # 3. Parse user instruction into numeric targets
    opt_targets = parse_instruction(custom_instruction)
    has_targets = any([opt_targets.fat_pct, opt_targets.msnf_pct, opt_targets.sugars_pct])

    if not has_targets:
        return {
            'success': True,
            'solver_status': 'N/A — Scale Only',
            'batch': batch.model_dump(),
            'scaled_recipe': scaled,
            'optimized_recipe': dict(scaled),
            'metrics_before': metrics_before.model_dump(),
            'metrics_after': metrics_before.model_dump(),
            'diffs': [],
            'warnings': warnings,
        }

    # 4. Run LP optimizer
    lp_result = run_lp_optimizer(scaled, opt_targets, mode=mode)
    proposed = lp_result['proposed_recipe']
    warnings.extend(lp_result.get('warnings', []))
    metrics_after = compute_recipe_metrics(proposed)

    # 5. Compute diffs
    diffs = []
    for name in scaled:
        orig_g = scaled[name]
        prop_g = proposed.get(name, 0)
        delta_g = prop_g - orig_g
        if abs(delta_g) > 0.5:
            delta_pct = (delta_g / orig_g * 100) if orig_g > 0 else 0
            diffs.append({
                'ingredient': name,
                'original_g': round(orig_g, 2),
                'proposed_g': round(prop_g, 2),
                'delta_g': round(delta_g, 2),
                'delta_pct': round(delta_pct, 2),
            })
    diffs.sort(key=lambda d: abs(d['delta_g']), reverse=True)

    return {
        'success': lp_result['success'],
        'solver_status': lp_result['solver_status'],
        'batch': batch.model_dump(),
        'scaled_recipe': scaled,
        'optimized_recipe': proposed,
        'metrics_before': metrics_before.model_dump(),
        'metrics_after': metrics_after.model_dump(),
        'diffs': diffs,
        'warnings': warnings,
    }

print('✅ optimizer_tool() ready.')

✅ optimizer_tool() ready.


In [22]:
# Cell 9 — Gemini Setup (google-genai SDK)
import os
import json as _json
from getpass import getpass
from google import genai
from google.genai import types

api_key = os.environ.get('GOOGLE_API_KEY') or getpass('🔑 Enter your Gemini API key: ')
os.environ['GOOGLE_API_KEY'] = api_key

client = genai.Client(api_key=api_key)

print('✅ Gemini client ready.')
print(f'   Key starts with: {api_key[:10]}...')

✅ Gemini client ready.
   Key starts with: AIzaSyATGG...


In [23]:
# Cell 9b — 🔍 DEBUG: Test the API with a tiny request
# Run ONLY this cell after Cell 9 to check if the API works at all.
# Try multiple models to see which ones work on your free tier.

test_models = ['gemini-2.0-flash-lite', 'gemini-2.0-flash', 'gemini-1.5-flash']

for model_name in test_models:
    try:
        response = client.models.generate_content(
            model=model_name,
            contents='Say hello in one word.',
        )
        print(f'✅ {model_name}: {response.text.strip()}')
    except Exception as e:
        err = str(e)
        if '429' in err:
            print(f'❌ {model_name}: RATE LIMITED (429)')
        elif 'API_KEY' in err:
            print(f'❌ {model_name}: API KEY INVALID')
        else:
            print(f'❌ {model_name}: {err[:100]}')

KeyboardInterrupt: 

In [24]:
# Cell 10 — Food Engineer PRE-optimization (interprets user request, modifies recipe)

FOOD_ENGINEER_PRE_PROMPT = """You are an expert food engineer specialising in ice cream,
gelato, kulfi, and frozen desserts.

You will receive:
  1. The user's request (e.g. "replace sugar with jaggery", "make it less creamy",
     "increase fat to 8%", etc.)
  2. The current recipe as a JSON list of {ingredient, quantity_g}.
  3. The available INGREDIENT_DB with all known ingredients and their properties.

Your job is to MODIFY the recipe based on the user's request BEFORE it goes to the
LP optimizer. You must reason and act:

  - REPLACEMENT requests (e.g. "replace X with Y"):
    Look for the replacement ingredient in INGREDIENT_DB.
    If found, swap it in with an appropriate quantity (use similar mass as the
    original, adjusted for concentration differences).
    Remove the original ingredient being replaced.

  - QUALITATIVE requests (e.g. "less creamy", "more icy", "lighter texture"):
    Reason about which ingredients affect that quality.
    Adjust quantities of existing ingredients, or swap ingredients, as appropriate.
    E.g. "less creamy" → reduce Cream 25% and/or increase Toned Milk 3%.

  - NUMERIC TARGET requests (e.g. "fat to 8%", "sugars to 20%"):
    Pass these through as-is — do NOT try to hit numeric targets yourself.
    The LP optimizer will handle the exact math.
    Just return the recipe unchanged for these.

RULES:
  - NEVER modify LOCKED ingredients (locked=True in the DB).
  - ONLY use ingredients that exist in INGREDIENT_DB.
  - Keep total recipe mass roughly the same (±10%).
  - Be practical: consider taste, texture, and production feasibility.

You MUST respond with ONLY valid JSON in this exact format, nothing else:
{
  "modified_recipe": [
    {"ingredient": "Name", "quantity_g": 1234.56},
    ...
  ],
  "changes_made": "Brief description of what you changed and why",
  "pass_to_optimizer": "The instruction string to pass to the optimizer (numeric targets only, or empty string if none)"
}
"""


def food_engineer_pre(user_prompt: str, recipe: list) -> dict:
    """
    Pre-optimization food engineer.
    Interprets the user's request, modifies the recipe (swaps, adjustments),
    and returns the modified recipe + instruction for the optimizer.
    """
    db_summary = {}
    for name, props in INGREDIENT_DB.items():
        db_summary[name] = {
            'fat_pct': props['fat_pct'],
            'msnf_pct': props['msnf_pct'],
            'sugars_pct': props['sugars_pct'],
            'water_pct': props['water_pct'],
            'category': props['category'],
            'locked': props['locked'],
        }

    prompt = f"""USER REQUEST:
{user_prompt}

CURRENT RECIPE:
{_json.dumps(recipe, indent=2)}

AVAILABLE INGREDIENTS (INGREDIENT_DB):
{_json.dumps(db_summary, indent=2)}

Analyze the request and return the modified recipe as JSON."""

    try:
        response = client.models.generate_content(
            model='gemini-3-flash-preview',
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=FOOD_ENGINEER_PRE_PROMPT,
            ),
        )
        raw = response.text.strip()

        # Strip markdown code fences if present
        if raw.startswith('```'):
            lines = raw.split('\n')
            lines = [l for l in lines if not l.strip().startswith('```')]
            raw = '\n'.join(lines)

        parsed = _json.loads(raw)
        return {
            'modified_recipe': parsed.get('modified_recipe', recipe),
            'changes_made': parsed.get('changes_made', 'No changes'),
            'pass_to_optimizer': parsed.get('pass_to_optimizer', user_prompt),
        }

    except Exception as e:
        print(f'⚠️ Food engineer pre-step failed: {e}')
        print('   Passing original recipe to optimizer unchanged.')
        return {
            'modified_recipe': recipe,
            'changes_made': f'Skipped (error: {e})',
            'pass_to_optimizer': user_prompt,
        }

print('✅ food_engineer_pre() ready.')

✅ food_engineer_pre() ready.


In [25]:
# Cell 11 — Food Scientist POST-optimization (reviews optimizer output)

FOOD_SCIENTIST_POST_PROMPT = """You are an expert food scientist specialising in ice cream,
gelato, kulfi, and frozen desserts.

You will receive:
  1. The user's original prompt / request.
  2. The LP optimizer output (a JSON with optimized_recipe, diffs, metrics, etc.).

Your job:
  - Reason about PRACTICALITY of the proposed changes.
  - Flag any issues: texture problems, taste imbalance, ingredient availability,
    cost concerns, or production feasibility.
  - If the user asked for ingredient REPLACEMENTS, comment on how the replacement
    affects the final product.
  - Be concise and actionable. No fluff.

You must NEVER:
  - Invent quantities not present in the optimizer output.
  - Suggest changes to LOCKED ingredients.
  - Contradict the solver's math — your role is to ADD practical insight on top.
"""


def food_scientist_reason(optimizer_output: dict, user_prompt: str) -> str:
    """
    Post-optimization food scientist.
    Reviews the optimizer output and gives practical food-science analysis.
    """
    prompt = f"""USER REQUEST:
{user_prompt}

OPTIMIZER OUTPUT:
{_json.dumps(optimizer_output, indent=2, default=str)}

Based on the above, give your practical food-science analysis.
Focus on: practicality of the changes, any texture/taste concerns,
and replacement suggestions if the user requested any."""

    try:
        response = client.models.generate_content(
            model='gemini-3-flash-preview',
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=FOOD_SCIENTIST_POST_PROMPT,
            ),
        )
        return response.text
    except Exception as e:
        return f"⚠️ Food scientist reasoning unavailable: {e}"

print('✅ food_scientist_reason() ready.')

✅ food_scientist_reason() ready.


In [26]:
# Cell 12 — Agent (Orchestrator)
# Flow: food_engineer_pre → optimizer_tool → food_scientist_reason

def run_agent(user_prompt: str, recipe: list, target_params: dict,
              mode: str = 'gelato') -> dict:
    """
    Main orchestrator.
    1. Food Engineer PRE  — interprets user request, modifies recipe.
    2. Optimizer Tool     — deterministic LP math on the modified recipe.
    3. Food Scientist POST — reviews optimizer output for practicality.

    Inputs:  user_prompt, recipe, target_params
    Returns: { optimized_recipe, metrics_before, metrics_after,
               diffs, engineer_changes, ai_analysis, warnings }
    """
    print('🔧 Step 1/3: Food Engineer analyzing request...')
    pre_result = food_engineer_pre(user_prompt, recipe)
    modified_recipe = pre_result['modified_recipe']
    engineer_changes = pre_result['changes_made']
    optimizer_instruction = pre_result['pass_to_optimizer']
    print(f'   ✅ Engineer changes: {engineer_changes}')

    print('⚙️  Step 2/3: Running LP optimizer...')
    opt_result = optimizer_tool(modified_recipe, target_params,
                               optimizer_instruction, mode)
    print(f'   ✅ Solver: {opt_result["solver_status"]}')

    print('🧪 Step 3/3: Food Scientist reviewing results...')
    ai_analysis = food_scientist_reason(opt_result, user_prompt)
    print('   ✅ Analysis complete.')

    return {
        'success': opt_result['success'],
        'solver_status': opt_result['solver_status'],
        'batch': opt_result['batch'],
        'optimized_recipe': opt_result['optimized_recipe'],
        'metrics_before': opt_result['metrics_before'],
        'metrics_after': opt_result['metrics_after'],
        'diffs': opt_result['diffs'],
        'engineer_changes': engineer_changes,
        'ai_analysis': ai_analysis,
        'warnings': opt_result['warnings'],
    }

print('✅ run_agent() ready.')

✅ run_agent() ready.


In [28]:
# Cell 13 — Test
TEST_RECIPE = [
    {'ingredient': 'Toned Milk 3%',           'quantity_g': 507970.39},
    {'ingredient': 'Cream 25%',               'quantity_g': 142300.70},
    {'ingredient': 'Sucrose/sugar',           'quantity_g': 101766.56},
    {'ingredient': 'Dextrose monohydrate',    'quantity_g': 15523.71},
    {'ingredient': 'Glucose Syrup (40-42DE)', 'quantity_g': 36221.99},
    {'ingredient': 'Condensed Milk Nestle',   'quantity_g': 15523.71},
    {'ingredient': 'Stabilizer',              'quantity_g': 5174.57},
    {'ingredient': 'Skimmed Milk Powder',     'quantity_g': 37946.85},
]

TEST_TARGET_PARAMS = {
    'lossPct': 5,
    'mixDensity': 1.04,
    'overrunPct': 27,
    'skuSizeLiters': 0.75,
    'targetVolumeLiters': 1000,
}

TEST_PROMPT = 'INSTEAD OF CONDENSED MILK USE AMUL BUFFALO MILK'

result = run_agent(
    user_prompt=TEST_PROMPT,
    recipe=TEST_RECIPE,
    target_params=TEST_TARGET_PARAMS,
    mode='gelato'
)

# Print results
print('\n=== ENGINEER CHANGES ===')
print(result['engineer_changes'])

print('\n=== OPTIMIZED RECIPE ===')
for name, grams in sorted(result['optimized_recipe'].items(), key=lambda x: -x[1]):
    print(f'  {name}: {grams:,.2f} g')

print(f'\n=== METRICS BEFORE ===')
for k, v in result['metrics_before'].items():
    print(f'  {k}: {v}')

print(f'\n=== METRICS AFTER ===')
for k, v in result['metrics_after'].items():
    print(f'  {k}: {v}')

print(f'\n=== DIFFS ===')
for d in result['diffs']:
    print(f"  {d['ingredient']}: {d['delta_g']:+,.2f} g ({d['delta_pct']:+.1f}%)")

print(f'\n=== AI ANALYSIS ===')
print(result['ai_analysis'])

if result['warnings']:
    print(f'\n=== WARNINGS ===')
    for w in result['warnings']:
        print(f'  {w}')

🔧 Step 1/3: Food Engineer analyzing request...
   ✅ Engineer changes: Replaced 'Condensed Milk Nestle' with 'Toned Milk 3%' by adding its mass (15523.71g) to the existing toned milk quantity. Since 'Amul Buffalo Milk' was not found in the INGREDIENT_DB, Toned Milk 3% was used as the closest available dairy substitute to fulfill the removal of condensed milk while maintaining total recipe mass. The optimizer will need to re-balance the fat and MSNF levels using Cream and Skimmed Milk Powder.
⚙️  Step 2/3: Running LP optimizer...
   ✅ Solver: N/A — Scale Only
🧪 Step 3/3: Food Scientist reviewing results...
   ✅ Analysis complete.

=== ENGINEER CHANGES ===
Replaced 'Condensed Milk Nestle' with 'Toned Milk 3%' by adding its mass (15523.71g) to the existing toned milk quantity. Since 'Amul Buffalo Milk' was not found in the INGREDIENT_DB, Toned Milk 3% was used as the closest available dairy substitute to fulfill the removal of condensed milk while maintaining total recipe mass. The optimiz

In [ ]:
# Cell 14 — Runtime Ingredient Extension
def add_ingredient(name, fat_pct=0, msnf_pct=0, sugars_pct=0, water_pct=0,
                   category='other', locked=False, note=''):
    """
    Add a custom ingredient to INGREDIENT_DB at runtime.
    Not persisted to disk (Supabase integration is a future step).
    """
    INGREDIENT_DB[name] = {
        'fat_pct': fat_pct, 'msnf_pct': msnf_pct,
        'sugars_pct': sugars_pct, 'water_pct': water_pct,
        'category': category, 'locked': locked, 'note': note,
    }
    lock_tag = ' 🔒' if locked else ''
    print(f"✅ Added '{name}' to INGREDIENT_DB{lock_tag}")
    print(f'   fat={fat_pct}%, msnf={msnf_pct}%, sugars={sugars_pct}%, water={water_pct}%')

# Example usage:
# add_ingredient('Butter 82%', fat_pct=82, msnf_pct=1.0, sugars_pct=0.5, water_pct=16, category='dairy')
print('✅ add_ingredient() ready.')

✅ add_ingredient() ready.
